# Malaria Detection: Try It Yourself
Loads the already-trained model straight from the shared Google Drive folder, so there is **no training** and nothing to upload. Just run the cells top to bottom, then upload cell images.

CPU runtime is fine. No GPU needed.

## 1. Load the model

In [ ]:
!pip install -q gdown

In [ ]:
import os, io, glob, shutil
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
import gdown

IMG_SIZE = 128
FOLDER_ID = "1aft0CmEm7xQJGtU22myvIUtXRCJO6rvH"   # shared Drive folder: malaria_project
MODEL_PATH = "malaria_model.keras"

The next cell downloads `malaria_model.keras` from the shared Drive folder. If that fails, it prints the exact error and then asks you to upload the file by hand.

**Most reliable option:** in Drive, right-click `malaria_model.keras` → Share → Copy link. The link looks like `drive.google.com/file/d/<FILE_ID>/view`. Paste that `<FILE_ID>` into `MODEL_FILE_ID` below and only that one file gets downloaded.

In [ ]:
MODEL_FILE_ID = ""   # optional, see note above

def fetch_model():
    if MODEL_FILE_ID:
        gdown.download(f"https://drive.google.com/uc?id={MODEL_FILE_ID}", MODEL_PATH, quiet=False)
        return
    gdown.download_folder(f"https://drive.google.com/drive/folders/{FOLDER_ID}",
                          output="malaria_project", quiet=False)
    hits = glob.glob("malaria_project/**/malaria_model.keras", recursive=True)
    if not hits:
        raise FileNotFoundError("malaria_model.keras not found in the Drive folder")
    shutil.copy(hits[0], MODEL_PATH)

if not os.path.exists(MODEL_PATH):
    try:
        fetch_model()
    except Exception as e:
        print("Auto download failed:", repr(e))

if not os.path.exists(MODEL_PATH):
    from google.colab import files
    print("Upload malaria_model.keras from your computer:")
    up = files.upload()
    MODEL_PATH = list(up.keys())[0]

print("Model file size: %.1f MB" % (os.path.getsize(MODEL_PATH) / 1e6))
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
print("Model loaded")

## 2. Test images
Click **Choose files** and select one or more cell images (png / jpg). Works best on single-cell crops like the ones in the dataset, not whole microscope slides.

In [ ]:
from google.colab import files

def predict_pil(img):
    img = img.convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    arr = np.array(img, dtype="float32")[None]
    return float(model.predict(arr, verbose=0)[0][0])

uploaded = files.upload()
n = len(uploaded)
fig, ax = plt.subplots(1, n, figsize=(3.5 * n, 4), squeeze=False)
for a, (name, data) in zip(ax[0], uploaded.items()):
    img = Image.open(io.BytesIO(data))
    prob = predict_pil(img)
    infected = prob >= 0.5
    a.imshow(img)
    a.set_title(f"{'Parasitized' if infected else 'Uninfected'}\n{(prob if infected else 1 - prob):.1%} confident",
                color="red" if infected else "green")
    a.axis("off")
plt.tight_layout(); plt.show()

## 3. Optional: web app
Gives a link with drag-and-drop upload, which is handy for a live demo. Run both cells, then open the **public URL** printed at the bottom.

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

def classify(img):
    prob = predict_pil(img)
    return {"Parasitized": prob, "Uninfected": 1 - prob}

gr.Interface(
    fn=classify,
    inputs=gr.Image(type="pil", label="Upload a cell image"),
    outputs=gr.Label(num_top_classes=2, label="Prediction"),
    title="Malaria Cell Detector",
    description="Upload a single blood smear cell image to check if it is parasitized.",
).launch(share=True)